# Skin Cancer Classification


Bu projede deri kanserini algılayan bir CNN modeli geliştireceğiz. Modeli kaydedip streamlit uygulamasına çevireceğiz ve hugging face de çalışır hle getireceğiz

In [1]:
import cv2
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
from keras.models import Sequential
from keras.layers import Conv2D,Dense, Flatten, Input, MaxPooling2D, Dropout, BatchNormalization, Reshape

import os


C:\Anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [3]:
import warnings 
warnings.filterwarnings('ignore')

In [4]:
labels=['Cancer', 'Non_Cancer']
img_path='Skin_Data/'


In [5]:
ls

 Volume in drive C has no label.
 Volume Serial Number is 787A-E273

 Directory of C:\Users\Kullan�c�\Documents\Yapay-Zeka\day12

16.02.2026  11:46    <DIR>          .
14.02.2026  13:27    <DIR>          ..
16.02.2026  07:53    <DIR>          .ipynb_checkpoints
14.02.2026  13:55             3.567 app.py
04.01.2026  19:04           141.824 cars.xls
16.02.2026  11:46         1.469.830 savasucagi.jpg
16.02.2026  10:15       165.522.260 skin_cancer_model.keras
16.02.2026  07:48    <DIR>          Skin_Data
16.02.2026  11:45            52.369 SkinCancerClassification.ipynb
               5 File(s)    167.189.850 bytes
               4 Dir(s)  255.863.664.640 bytes free


In [6]:
img_list=[]
label_list=[]
for label in labels:
    for img_file in os listdir(img_path+label)
        img_list.append(img_path+'/'+label)
        label_list.append(label)

SyntaxError: invalid syntax (2008212102.py, line 4)

In [ ]:
import os

labels = ['Cancer', 'Non_Cancer']
img_path = "Skin_Data"   # klasörün adı (ekranda görünen)

img_list = []
label_list = []

for label in labels:
    folder = os.path.join(img_path, label)  # Skin_Data/Cancer gibi
    for img_file in os.listdir(folder):
        img_list.append(os.path.join(folder, img_file))  # dosyanın tam yolu
        label_list.append(label)

print("Toplam görsel:", len(img_list))
print("Toplam etiket:", len(label_list))
print("Örnek dosya:", img_list[0] if img_list else "Boş")


In [ ]:
df=pd.DataFrame({'img':img_list, 'label':label_list})

In [ ]:
df


In [ ]:
d=({'Cancer':1, 'Non_Cancer':0})
df['label_encoded']=df['label'].map(d)

In [ ]:
df

In [ ]:
x=[]
for img in df['img']:
    img=cv2.imread(str(img))
    img=cv2.resize(img,(170,170))
    img=img/255.0
    x.append(img)

In [ ]:
x=np.array(x)

In [ ]:
x.shape

In [ ]:
y=df['label_encoded']

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y, random_state=42, test_size=0.20)

In [ ]:
y_train=np.array(y_train, dtype=np.int32)
y_test=np.array(y_test, dtype=np.int32)

In [ ]:
# --- MODEL (Sequential sonrası) ---

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense

model = Sequential()
model.add(Input(shape=(170, 170, 3)))

model.add(Conv2D(32, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())

model.add(Dense(128, activation='relu'))      # hidden layer
model.add(Dense(1, activation='sigmoid'))     # binary output (0/1)

# Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
history = model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=20,
    verbose=1
)



In [ ]:
model.save("skin_cancer_model.keras")

# Transfer Learning

Akıllı insan aklını kullanandır. Daha akıllı insan başkalarının aklını da kullanandır.

In [ ]:
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential

data_dir = 'Skin_Data'
img_width, img_height = 224, 224

# Training + Validation generator (aynı generator, validation_split ile bölünüyor)
train_datagen = ImageDataGenerator(rescale=1/255, validation_split=0.20)

train_generator = train_datagen.flow_from_directory(
    directory=data_dir,
    target_size=(img_width, img_height),
    class_mode='binary',
    subset='training'
)

test_datagen = ImageDataGenerator(rescale=1/255)

test_generator = train_datagen.flow_from_directory(
    directory=data_dir,
    target_size=(img_width, img_height),
    class_mode='binary',
    subset='validation'
)

# VGG16 taban modeli (Imagenet ağırlıkları ile, üstteki classifier kısmı yok)
base_model = VGG16(
    weights='imagenet',
    input_shape=(img_width, img_height, 3),
    include_top=False
)

# Modeli oluştur
model = Sequential()
model.add(base_model)

# Base model katmanlarını dondur (transfer learning - feature extractor gibi kullan)
for layer in base_model.layers:
    layer.trainable = False


In [ ]:
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense

data_dir = 'Skin_Data'
img_width, img_height = 224, 224

train_datagen = ImageDataGenerator(rescale=1/255, validation_split=0.20)

train_datagenerator = train_datagen.flow_from_directory(
    directory=data_dir,
    target_size=(img_width, img_height),
    class_mode='binary',
    subset='training'
)

test_datagen = ImageDataGenerator(rescale=1/255)

test_datagenerator = train_datagen.flow_from_directory(
    directory=data_dir,
    target_size=(img_width, img_height),
    class_mode='binary',
    subset='validation'
)

base_model = VGG16(
    weights='imagenet',
    input_shape=(img_width, img_height, 3),
    include_top=False
)

model = Sequential()
model.add(base_model)

for layer in base_model.layers:
    layer.trainable = False

model.add(Flatten())
model.add(Dense(1024, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_datagenerator,
    epochs=10,
    validation_data=test_datagenerator
)



In [ ]:
model.save('skin_cancer_TL_keras')

In [ ]:
from tensorflow.keras.applications.resnet50 import ResNet50


In [ ]:
from tensorflow.keras.preprocessing import image



In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input, decode_predictions


In [ ]:
from IPython.display import Image


In [7]:
# ===============================
# IMPORTLAR
# ===============================
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input, decode_predictions
from IPython.display import Image
import numpy as np

# ===============================
# GÖRSELİ GÖSTER
# ===============================
Image('savasucagi.jpg')

# ===============================
# GÖRSELİ YÜKLE ve HAZIRLA
# ===============================
img = image.load_img('savasucagi.jpg', target_size=(224, 224))
img = image.img_to_array(img)
img = np.expand_dims(img, axis=0)
img = preprocess_input(img)

# ===============================
# MODELİ YÜKLE (ImageNet ağırlıkları)
# ===============================
model = ResNet50(weights='imagenet')

# ===============================
# TAHMİN YAP
# ===============================
pred = model.predict(img)

# ===============================
# SONUCU DECODE ET (EN YÜKSEK 1 TAHMİN)
# ===============================
decode_predictions(pred, top=1)


102967424/102967424 ━━━━━━━━━━━━━━━━━━━━ 14s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


[[('n04552348', 'warplane', np.float32(0.7799485))]]

In [8]:
pip install python-doctr[torch]

   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ----- ---------------------------------- 2.1/16.4 MB 9.5 MB/s eta 0:00:02
   --------- ------------------------------ 3.9/16.4 MB 8.9 MB/s eta 0:00:02
   ------------ --------------------------- 5.2/16.4 MB 7.9 MB/s eta 0:00:02
   ----------------- ---------------------- 7.3/16.4 MB 8.5 MB/s eta 0:00:02
   ---------------------- ----------------- 9.2/16.4 MB 8.5 MB/s eta 0:00:01
   --------------------------- ------------ 11.3/16.4 MB 8.7 MB/s eta 0:00:01
   ------------------------------- -------- 13.1/16.4 MB 8.7 MB/s eta 0:00:01
   ------------------------------------- -- 15.2/16.4 MB 8.8 MB/s eta 0:00:01
   ---------------------------------------- 16.4/16.4 MB 8.6 MB/s  0:00:01
   ---------------------------------------- 0.0/3.1 MB ? eta -:--:--
   ----------------------- ---------------- 1.8/3.1 MB 9.9 MB/s eta 0:00:01
   ---------------------------------------- 3.1/3.1 MB 7.7 MB/s  0:00:00
   ---------------